# N-Candidate Solver — Drawer Slides (N=1)

The simplest case: a **single-product family**. No Cartesian product — the solver evaluates each slide individually against requirements.

**Compare with:**
- `v2_hinge_n_candidate_demo.ipynb` — N=2 (concealed hinges)
- `v2_n_candidate_demo.ipynb` — N=3 (LED lighting)

In [ ]:
import asyncio, sys
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

import sys
from pathlib import Path

# Find project root
_project_root = Path.cwd()
while _project_root != _project_root.parent:
    if (_project_root / "sample-data").exists() and (_project_root / "engine_v2").exists():
        break
    _project_root = _project_root.parent
sys.path.insert(0, str(_project_root))
DATA_DIR = _project_root / "sample-data"

from engine_v2.core.solver_n import NCandidateSolver
from engine_v2.families.drawer_slide.config import SLIDE_N_CONFIG
from engine_v2.families.drawer_slide.loader import load_from_json
from engine_v2.families.drawer_slide.models import (
    ExtensionType, SlideCloseType, SlideMountType, SlideRequirements,
)

all_slides = load_from_json(DATA_DIR)
solver = NCandidateSolver(config=SLIDE_N_CONFIG, product_lists={"slide": all_slides})

print(f"Loaded {len(all_slides)} drawer slides")
print(f"Roles: {SLIDE_N_CONFIG.role_names}  (N={len(SLIDE_N_CONFIG.roles)})")
print(f"Rules: {len(SLIDE_N_CONFIG.rules)}")

## 1. Product Catalog

In [ ]:
print(f"{'SKU':<16} {'Brand':<8} {'Series':<26} {'Length':>6} {'Load':>6} {'Extension':<16} {'Mount':<14} {'Close':<12} {'Price':>7}")
print("-" * 110)
for s in all_slides:
    print(f"{s.sku:<16} {s.brand:<8} {s.series:<26} {s.slide_length_mm:>4}mm {s.max_load_kg:>4.0f}kg {s.extension_type.value:<16} {s.mount_type.value:<14} {s.close_type.value:<12} ${s.price_usd:.2f}")

## 2. Scenarios

In [ ]:
# Scenario 1: Standard kitchen drawer
req = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=15.0)
result = solver.solve_with_explanation(req)

print(f"Status: {result['status']}")
print(f"{result['message']}\n")

if result["status"] == "solved":
    rec = result["recommended"]
    print(f"Recommended: {rec['candidates']['slide']['sku']}")
    print(f"Price: ${rec['total_price_usd']:.2f}\n")
    print(f"Constraint trace ({len(rec['constraint_trace'])} rules):")
    for rule in rec["constraint_trace"]:
        icon = "PASS" if rule["passed"] else "FAIL"
        print(f"  [{icon}] {rule['rule']} {rule['name']}: {rule['detail']}")
    print(f"\n+ {len(result['alternatives'])} alternative(s)")

In [ ]:
# Scenario 2: Heavy-duty
results = solver.solve(SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=42.0))
print(f"Found {len(results)} slide(s) for 42kg drawer:")
for config in results:
    s = config.candidates["slide"]
    print(f"  {s.sku} - {s.brand} {s.series}, {s.max_load_kg}kg rating")

In [ ]:
# Scenario 3: Impossible - cabinet too shallow
fail = solver.solve_with_explanation(SlideRequirements(cabinet_depth_mm=300, drawer_weight_kg=5.0))
print(f"Status: {fail['status']}")
print(f"{fail['message']}\n")
if fail.get("failed_rules"):
    for fr in fail["failed_rules"]:
        print(f"  [{fr['rule']}] {fr['name']}: {fr['detail']}")

## 3. Exhaustive Evaluation

With N=1, every slide is evaluated individually. No Cartesian product.

In [ ]:
solver.config.early_termination = False
baseline = SlideRequirements(cabinet_depth_mm=550, drawer_weight_kg=15.0)

print(f"{'SKU':<16} {'Valid':>5}  Rule Results")
print("-" * 90)
for slide in all_slides:
    config = solver.evaluate({"slide": slide}, baseline)
    status = "YES" if config.valid else "NO"
    rules = "  ".join(f"{r.rule_id}:{'PASS' if r.passed else 'FAIL'}" for r in config.rule_results)
    print(f"{slide.sku:<16} {status:>5}  {rules}")

solver.config.early_termination = True
print(f"\nTotal: {len(all_slides)} slides x {len(SLIDE_N_CONFIG.rules)} rules = {len(all_slides) * len(SLIDE_N_CONFIG.rules)} evaluations")